In [ ]:
import os
import sys
os.chdir("/workspaces/dev")
sys.path.append("/workspaces/dev/modules/python-utils")
sys.path.append("/workspaces/dev/modules/ai-utils")

In [ ]:
import librosa
import numpy as np
from pathlib import Path
from IPython.display import Audio

In [ ]:
from rt_whisper import streamers
from rt_whisper.data import Param, Result
from sj_utils.audio_utils import segment_audio

In [ ]:
SEED = 42
SAMPLE_RATE = 16000
SOURCE = "/workspaces/dev/.data/news_with_English.mp3"

In [ ]:
src = Path(SOURCE)
src.exists()

In [ ]:
np.random.seed(SEED)
audio, sr = librosa.load(src, sr=SAMPLE_RATE)
segments = segment_audio(audio)
Audio(audio, rate=sr)

In [ ]:
token_streamer = streamers.get_token_streamer_with_vad_v2()

In [ ]:
raise Exception("stop")

In [ ]:
segment_idx = 0
completed = []
param = Param()
param.offset = 0

In [ ]:
segment = segments[segment_idx]
segment_idx += 1

param.chunk = segment

result:Result = token_streamer.process(param)
completed.extend(result.completed)

# print(f"{segment_idx}" + "--" * 20)
# print(
#     [(v.tokens[0].start, v.tokens[-1].end, v.lang, v.text)
#     for v in result.completed]
# )
# print(
#     [(v.tokens[0].start, v.tokens[-1].end, v.lang, v.text)
#     for v in result.candidate]
# )

param.update(result, update_prompt=False)


In [ ]:
from rt_whisper.processors.asr import ASRState
Audio(result.context_dict[ASRState].chunk, rate=SAMPLE_RATE)

In [ ]:
from rt_whisper.processors.vad import VADState
# VAD
Audio(result.context_dict[VADState].chunk, rate=SAMPLE_RATE)

In [ ]:
start = result.completed[0].tokens[0].start
end = result.completed[0].tokens[-1].end
Audio(audio[start: end], rate=SAMPLE_RATE)

In [ ]:
start = result.completed[1].tokens[0].start
end = result.completed[1].tokens[-1].end
Audio(audio[start: end], rate=SAMPLE_RATE)

In [ ]:
Audio(result.recycles["vad"].vad_chunk, rate=SAMPLE_RATE)
# len(result.recycles["vad"].vad_chunk), len(segment)

In [ ]:
for i, segment in enumerate(segments):
    param.chunk = segment

    result:Result = token_streamer.process(param)
    completed.extend(result.completed)

    # print(f"{i}" + "--" * 20)
    # print([(v.lang, v.text) for v in completed])
    # print([(v.lang, v.text) for v in result.completed])
    # print([(v.lang, v.text) for v in result.candidate])
    # print([(v.lang, v.text) for v in result.prev_completed_tokens if v.is_word])
    # print([(v.lang, v.text) for v in result.prev_candidate_tokens if v.is_word])

    param.update(result, update_prompt=True)

In [ ]:
for v in completed:
    print(v.lang, v.text)